<a href="https://colab.research.google.com/github/CefasRepRes/shipwreck-oil-detection-colab/blob/main/CefasCobblesTools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def install_requirements():

  global np, seg, sam, predictor, unet, si, torchvision, torch, SAM2ImagePredictor, build_sam2, pd, pd, cv2, measure, reload, trange, tqdm, pickle, Path, predictor, sam, device, sam_checkpoint
  from google.colab import output
  output.enable_custom_widget_manager()
  %matplotlib ipympl
  import torch
  import torchvision
  print("PyTorch version:", torch.__version__)
  print("Torchvision version:", torchvision.__version__)
  print("CUDA is available:", torch.cuda.is_available())
  import sys
  !{sys.executable} -m pip install opencv-python matplotlib
  !{sys.executable} -m pip install sam2
  !{sys.executable} -m pip install segmenteverygrain
  !{sys.executable} -m pip install rtree
  !wget https://huggingface.co/facebook/sam2.1-hiera-large/resolve/main/sam2.1_hiera_large.pt
  from sam2.build_sam import build_sam2
  from sam2.sam2_image_predictor import SAM2ImagePredictor
  sam_checkpoint = "sam2.1_hiera_large.pt"
  device = "cuda"
  sam = build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", sam_checkpoint, device=device)
  predictor = SAM2ImagePredictor(sam)
  import pandas as pd
  import numpy as np
  import matplotlib.pyplot as plt
  import cv2
  from skimage import measure
  from importlib import reload
  import segmenteverygrain as seg
  import segmenteverygrain.interactions as si
  from tqdm import trange, tqdm
  import pickle
  from pathlib import Path
  !wget "https://raw.githubusercontent.com/zsylvester/segmenteverygrain/main/models/seg_model_smooth_labels.pt"
  # U-Net model (PyTorch)
  unet = seg.load_unet("seg_model_smooth_labels.pt")
  !wget -O WWBF_CEND0121_WWBF008_STN_182_A1_006.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF008_STN_182_A1_006.JPG
  !wget -O WWBF_CEND0121_WWBF008_STN_182_A1_011.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF008_STN_182_A1_011.JPG
  !wget -O WWBF_CEND0121_WWBF008_STN_182_A1_020.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF008_STN_182_A1_020.JPG
  !wget -O WWBF_CEND0121_WWBF013_STN_180_A1_005.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF013_STN_180_A1_005.JPG
  !wget -O WWBF_CEND0121_WWBF013_STN_180_A1_023.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF013_STN_180_A1_023.JPG
  !wget -O WWBF_CEND0121_WWBF019_STN_174_A1_022.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF019_STN_174_A1_022.JPG


Code to create the image dropdown box

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
def Select_image():
  global IMGdropdown, min_area_px, dbs__max_dist
  # Folder containing the images
  image_dir = Path("/content")

  # Find JPG files, ignoring case
  image_files = sorted([
     f for f in image_dir.iterdir()
      if f.is_file() and f.suffix.lower() == ".jpg"
  ])

  # Dropdown shows filename without extension
  IMGdropdown = widgets.Dropdown(
     options=[f.stem for f in image_files],
     description="Select image:"
  )
  # Slider for pixel filter
  min_area_px = widgets.IntSlider(value=250, min=50, max=750, step=1, description='Min pixel area', readout=True, orientation='horizontal', layout=widgets.Layout(width='400px'), style={'description_width': 'auto'})
  dbs__max_dist = widgets.IntSlider(value=30, min=10, max=500, step=1, description='Prompt distance', readout=True, orientation='horizontal', layout=widgets.Layout(width='400px'), style={'description_width': 'auto'})

  return display(IMGdropdown, min_area_px,dbs__max_dist)


Dropdown(description='Image:', options=('WWBF_CEND0121_WWBF008_STN_182_A1_006', 'WWBF_CEND0121_WWBF008_STN_182…

In [ ]:
def Run_segmentation():
  global all_grains, labels, mask_all, grain_data, fig, ax
  # replace this with the path to your image:
  fname = f'{IMGdropdown.value}.JPG'
  image = np.array(seg.load_img(fname))
  image_pred = seg.predict_image(image, unet, I=256)

  # decreasing the 'dbs_max_dist' parameter results in more SAM prompts
  # (and longer processing times):
  labels, coords = seg.label_grains(image, image_pred, dbs_max_dist=dbs__max_dist.value)

  # SAM segmentation, using the point prompts from the Unet:
  all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
    sam,
    image,
    image_pred,
    coords,
    labels,
    min_area=min_area_px.value,
    plot_image=True,
    remove_edge_grains=False,
    remove_large_objects=False,
  )

In [ ]:
def save_segmented_outputs():
  # Convert polygons to Grain objects
  grains = si.polygons_to_grains(all_grains, image=image)
  for g in tqdm(grains, desc='Measuring grains'):
    g.measure()

  pkl_filename = Path(fname).with_suffix(".pkl")
  with open(pkl_filename, "wb") as f:
    pickle.dump((
            all_grains,
            labels,
            mask_all,
            grain_data,),f)
  print(f"Saved segmentation to: {pkl_filename}")

Measuring grains: 100%|██████████| 825/825 [00:00<00:00, 1765.12it/s]

Saved segmentation to: WWBF_CEND0121_WWBF008_STN_182_A1_006.pkl
